## 1. Setup & Configuration
Spark session + MySQL JDBC connection settings

In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

In [ ]:
MYSQL_HOST = "localhost"
MYSQL_PORT = "3306"
MYSQL_DB = "olist_raw"
MYSQL_USER = "student"
MYSQL_PASSWORD = "student"

JDBC_URL = f"jdbc:mysql://{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}?useSSL=false&allowPublicKeyRetrieval=true&zeroDateTimeBehavior=convertToNull"

JDBC_PROPERTIES = {
    "user": MYSQL_USER,
    "password": MYSQL_PASSWORD,
    "driver": "com.mysql.jdbc.Driver"
}

In [8]:
spark = (
    SparkSession.builder
    .appName("Cleaning_DimProduct")
    .config("spark.jars", "/usr/local/spark3/spark-3.1.2-bin-hadoop3.2/jars/mysql-connector-java-5.1.47.jar")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [9]:
def read_table(table_name: str):
    
    return spark.read.jdbc(url=JDBC_URL, table=table_name, properties=JDBC_PROPERTIES)

## 2. Ingestion — Reading Raw Tables from MySQL

In [11]:
raw_customers = read_table("raw_customers")
raw_customers.show(5)

+--------------------+--------------------+------------------------+-------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+--------------------+--------------------+------------------------+-------------+--------------+
|00012a2ce6f8dcda2...|248ffe10d632bebe4...|                    6273|       osasco|            SP|
|000161a058600d590...|b0015e09bb4b6e47c...|                   35550|  itapecerica|            MG|
|0001fd6190edaaf88...|94b11d37cd61cb299...|                   29830| nova venecia|            ES|
|0002414f953443074...|4893ad4ea28b2c5b3...|                   39664|     mendonca|            MG|
|000379cdec6255224...|0b83f73b19c2019e1...|                    4841|    sao paulo|            SP|
+--------------------+--------------------+------------------------+-------------+--------------+
only showing top 5 rows



In [35]:
raw_products = read_table("raw_products")
raw_orders = read_table("raw_orders")
raw_order_items = read_table("raw_order_items")
raw_order_payments = read_table("raw_order_payments")

print("")
for name, df in [
    ("customers", raw_customers),
    ("products", raw_products),
    ("orders", raw_orders),
    ("order_items", raw_order_items),
    ("order_payments", raw_order_payments),
]:
    print(f"{name}: {df.count()}")


customers: 99441
products: 32951
orders: 99441
order_items: 225300
order_payments: 207772


## 3. Landing Raw Data on HDFS (Raw Zone)

In [37]:
RAW_ZONE_PATH = "/raw_zone"

raw_customers.write.mode("overwrite").parquet(f"{RAW_ZONE_PATH}/customers")
raw_products.write.mode("overwrite").parquet(f"{RAW_ZONE_PATH}/products")
raw_orders.write.mode("overwrite").parquet(f"{RAW_ZONE_PATH}/orders")
raw_order_items.write.mode("overwrite").parquet(f"{RAW_ZONE_PATH}/order_items")
raw_order_payments.write.mode("overwrite").parquet(f"{RAW_ZONE_PATH}/order_payments")


In [38]:
raw_customers = spark.read.parquet(f"{RAW_ZONE_PATH}/customers")
raw_products = spark.read.parquet(f"{RAW_ZONE_PATH}/products")
raw_orders = spark.read.parquet(f"{RAW_ZONE_PATH}/orders")
raw_order_items = spark.read.parquet(f"{RAW_ZONE_PATH}/order_items")
raw_order_payments = spark.read.parquet(f"{RAW_ZONE_PATH}/order_payments")

raw_customers.show(3)

+--------------------+--------------------+------------------------+-------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+--------------------+--------------------+------------------------+-------------+--------------+
|00012a2ce6f8dcda2...|248ffe10d632bebe4...|                    6273|       osasco|            SP|
|000161a058600d590...|b0015e09bb4b6e47c...|                   35550|  itapecerica|            MG|
|0001fd6190edaaf88...|94b11d37cd61cb299...|                   29830| nova venecia|            ES|
+--------------------+--------------------+------------------------+-------------+--------------+
only showing top 3 rows



## 4. Data Cleaning & Validation

In [13]:
customers_clean = (
    raw_customers
    .dropDuplicates(["customer_id"])
    .filter(F.col("customer_id").isNotNull())
    .withColumn("customer_city", F.trim(F.lower(F.col("customer_city"))))
    .withColumn("customer_state", F.trim(F.upper(F.col("customer_state"))))
)

print(f"{raw_customers.count()}")
print(f"{customers_clean.count()}")
customers_clean.show(5)

99441


99441


+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|01d190d14b00073f7...|2e5dcf79b225e8d16...|                    2925|           sao paulo|            SP|
|03a7750fc7a7bfbd7...|ae7e471f70f6fb521...|                   83430|campina grande do...|            PR|
|04495037fc6899faf...|c611b2ddcec542760...|                   15953|             botelho|            SP|
|04b7d26bde4f2d2fe...|9ef6d1d9fdc6511e4...|                   38067|             uberaba|            MG|
|04cef6b920c0d8f16...|6d0b86c615a3aa7ef...|                    3183|           sao paulo|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows



In [14]:
products_clean = (
    raw_products
    .dropDuplicates(["product_id"])
    .filter(F.col("product_id").isNotNull())
    .withColumn(
        "product_category_name",
        F.when(F.col("product_category_name").isNull(), "unknown")
         .otherwise(F.trim(F.lower(F.col("product_category_name"))))
    )
    .withColumn("product_weight_g",
                F.when(F.col("product_weight_g") > 0, F.col("product_weight_g")))
    .withColumn("product_length_cm",
                F.when(F.col("product_length_cm") > 0, F.col("product_length_cm")))
    .withColumn("product_height_cm",
                F.when(F.col("product_height_cm") > 0, F.col("product_height_cm")))
    .withColumn("product_width_cm",
                F.when(F.col("product_width_cm") > 0, F.col("product_width_cm")))
)

print(f"Before: {raw_products.count()}, After: {products_clean.count()}")
products_clean.show(5)

Before: 32951, After: 32951
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|00e4ded51458037ec...| informatica_acess...|                 25|                       978|                 3|            1400|               25|               15|              25|
|03d7ad0ce97624c93...|           perfumaria|                 48|                       259|                 1|             249|               19|               13|              15|
|04008f5a086abe248...|      cama_mesa_banho|                 41|   

In [28]:
orders_clean = (
    raw_orders
    .dropDuplicates(["order_id"])
    .filter(
        F.col("order_id").isNotNull()
        & F.col("customer_id").isNotNull()
        & F.col("order_purchase_timestamp").isNotNull()  
    )
)

In [16]:
order_items_clean = (
    raw_order_items
    .filter(
        F.col("order_id").isNotNull()
        & F.col("product_id").isNotNull()
        & F.col("price").isNotNull()
        & (F.col("price") > 0)          
    )
    .dropDuplicates(["order_id", "order_item_id"])
)

In [17]:
order_payments_clean = (
    raw_order_payments
    .filter(
        F.col("order_id").isNotNull()
        & F.col("payment_value").isNotNull()
        & (F.col("payment_value") >= 0)
    )
)

In [18]:
print("\n")
for name, df in [
    ("customers_clean", customers_clean),
    ("products_clean", products_clean),
    ("orders_clean", orders_clean),
    ("order_items_clean", order_items_clean),
    ("order_payments_clean", order_payments_clean),
]:
    print(f"{name}: {df.count()}")


customers_clean: 99441


products_clean: 32951


orders_clean: 99441


order_items_clean: 112650
order_payments_clean: 207772


## 5. Building Dim_Product (SCD Type 2)

In [19]:
price_history = (
    order_items_clean
    .join(orders_clean.select("order_id", "order_purchase_timestamp"), on="order_id", how="inner")
    .select(
        "product_id",
        F.col("order_purchase_timestamp").cast("date").alias("price_date"),
        "price"
    )
)

In [20]:
price_history_dedup= price_history.dropDuplicates(["product_id", "price_date", "price"])

In [21]:
window_by_product_date = Window.partitionBy("product_id").orderBy("price_date")

price_with_prev = price_history_dedup.withColumn(
    "prev_price", F.lag("price").over(window_by_product_date)
)

In [22]:
price_changes = price_with_prev.filter(
    F.col("prev_price").isNull() | (F.col("price") != F.col("prev_price"))
).drop("prev_price")

In [23]:
window_next_change = Window.partitionBy("product_id").orderBy("price_date")

product_price_scd = (
    price_changes
    .withColumn("dw_start_date", F.col("price_date"))
    .withColumn(
        "next_start_date",
        F.lead("price_date").over(window_next_change)
    )
    .withColumn(
        "dw_end_date",
        F.when(F.col("next_start_date").isNotNull(), F.date_sub(F.col("next_start_date"), 1))
         .otherwise(F.lit("9999-12-31").cast("date"))
    )
    .select("product_id", F.col("price").alias("product_price"), "dw_start_date", "dw_end_date")
)

In [24]:
dim_product = (
    product_price_scd
    .join(products_clean, on="product_id", how="left")
    .withColumn("product_key", F.monotonically_increasing_id())  # Surrogate Key
    .select(
        "product_key",
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_price",
        "dw_start_date",
        "dw_end_date",
    )
    .orderBy("product_id", "dw_start_date")
)

print("\n")
dim_product.show(10, truncate=False)
print(f"Number of rows of Dim_Product: {dim_product.count()}")
print(f"Number of unique products: {dim_product.select('product_id').distinct().count()}")


+-------------+--------------------------------+---------------------+----------------+-----------------+-----------------+----------------+-------------+-------------+-----------+
|product_key  |product_id                      |product_category_name|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_price|dw_start_date|dw_end_date|
+-------------+--------------------------------+---------------------+----------------+-----------------+-----------------+----------------+-------------+-------------+-----------+
|8589934592   |00066f42aeeb9f3007548bb9d3f33c38|perfumaria           |300             |20               |16               |16              |101.65       |2018-05-20   |9999-12-31 |
|541165879296 |00088930e925c41fd95ebfe695fd2655|automotivo           |1225            |55               |10               |26              |129.90       |2017-12-12   |9999-12-31 |
|1357209665536|0009406fd7479715e4bef61dd91f2462|cama_mesa_banho      |300             |45      

Number of rows of Dim_Product: 43197


Number of unique products: 32951


## 6. Exporting Final Outputs

In [40]:
customers_clean.write.mode("overwrite").parquet(
    "/user/student/cleaned_data/customers_clean"
)

orders_clean.write.mode("overwrite").parquet(
    "/user/student/cleaned_data/orders_clean"
)

order_items_clean.write.mode("overwrite").parquet(
    "/user/student/cleaned_data/order_items_clean"
)

order_payments_clean.write.mode("overwrite").parquet(
    "/user/student/cleaned_data/order_payments_clean"
)

products_clean.write.mode("overwrite").parquet(
    "/user/student/cleaned_data/products_clean"
)

dim_product.write.mode("overwrite").parquet(
    "/user/student/cleaned_data/dim_product"
)

In [41]:
spark.read.parquet(f"/user/student/cleaned_data/customers_clean").show(5)


+--------------------+--------------------+------------------------+--------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix| customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------+--------------+
|0063913c2f1878cc4...|3c4abb94aa82c0be6...|                   13178|        sumare|            SP|
|00e8bdabd8d9dec77...|3fcda7fdc267ed83f...|                   36420|   ouro branco|            MG|
|02405cd33ab625b31...|9696892d2f285e3e3...|                   30140|belo horizonte|            MG|
|0263aeaed91e6e437...|71cb999c1d3226677...|                    2349|     sao paulo|            SP|
|035cbd7a946d33043...|ca994abc57b0bd798...|                   90570|  porto alegre|            RS|
+--------------------+--------------------+------------------------+--------------+--------------+
only showing top 5 rows

